# XLK Clean-Data Random Forest Baseline

This notebook establishes a pre-specified clean-data Random Forest baseline using the same predictors, target definitions, chronological assignments and validation principles as the completed Logistic Regression baseline. No feature noise, scaling, feature removal, threshold optimisation or hyperparameter tuning is undertaken.

In [ ]:
from pathlib import Path
import os

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'src').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise RuntimeError('Run this notebook from inside the cloned repository.')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.venv' / '.matplotlib'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay, f1_score, precision_score,
    recall_score, roc_auc_score,
)
from sklearn.model_selection import TimeSeriesSplit

DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'xlk_feature_dataset.csv'
DICTIONARY_PATH = PROJECT_ROOT / 'outputs' / 'tables' / 'xlk_feature_dictionary.csv'
LOGISTIC_TEST_PATH = PROJECT_ROOT / 'outputs' / 'tables' / 'xlk_logistic_baseline_metrics.csv'
LOGISTIC_CV_PATH = PROJECT_ROOT / 'outputs' / 'tables' / 'xlk_logistic_cv_metrics.csv'
RF_CV_PATH = PROJECT_ROOT / 'outputs' / 'tables' / 'xlk_random_forest_cv_metrics.csv'
RF_TEST_PATH = PROJECT_ROOT / 'outputs' / 'tables' / 'xlk_random_forest_test_metrics.csv'
IMPORTANCE_PATH = PROJECT_ROOT / 'outputs' / 'tables' / 'xlk_random_forest_feature_importance.csv'
COMPARISON_PATH = PROJECT_ROOT / 'outputs' / 'tables' / 'xlk_clean_model_comparison.csv'
CONFUSION_PATH = PROJECT_ROOT / 'outputs' / 'figures' / 'xlk_random_forest_confusion_matrix.png'
COMPARISON_FIGURE_PATH = PROJECT_ROOT / 'outputs' / 'figures' / 'xlk_clean_model_comparison.png'
IMPORTANCE_FIGURE_PATH = PROJECT_ROOT / 'outputs' / 'figures' / 'xlk_random_forest_feature_importance.png'

for path in [DATA_PATH, DICTIONARY_PATH, LOGISTIC_TEST_PATH, LOGISTIC_CV_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f'Required project input is missing: {path}')
sns.set_theme(style='whitegrid')
print(f'Confirmed project root: {PROJECT_ROOT}')

## 1. Inputs and preserved assignments

The 11 predictor names are read from the feature dictionary. Existing target values and sample assignments are preserved without recalculating the fixed target, split or purge. Logistic Regression results are read solely for later comparison.

In [ ]:
data = pd.read_csv(DATA_PATH, parse_dates=['Date'])
feature_dictionary = pd.read_csv(DICTIONARY_PATH)
existing_test_metrics = pd.read_csv(LOGISTIC_TEST_PATH)
existing_logistic_cv = pd.read_csv(LOGISTIC_CV_PATH)
predictors = feature_dictionary['feature_name'].tolist()
target = 'high_volatility'

assert len(predictors) == 11 and len(set(predictors)) == 11, 'Exactly 11 documented predictors are required.'
assert set(predictors).issubset(data.columns), 'Every documented predictor must be present.'
assert data['Date'].is_monotonic_increasing and data['Date'].is_unique, 'Dates must be increasing and unique.'
assert not data[predictors + [target, 'future_rv_5d', 'sample_period']].isna().any().any(), \
    'Model inputs, targets and labels must be complete.'
assert np.isfinite(data[predictors].to_numpy(dtype=float)).all(), 'Predictors must be finite.'

original_assignments = data[['Date', target, 'sample_period']].copy(deep=True)
train_data = data.loc[data['sample_period'].eq('train')].copy()
test_data = data.loc[data['sample_period'].eq('test')].copy()
X_train = train_data[predictors]
y_train = train_data[target].astype(int)
X_test = test_data[predictors]
y_test = test_data[target].astype(int)

purged_dates = pd.to_datetime(['2021-03-03', '2021-03-04', '2021-03-05', '2021-03-08', '2021-03-09'])
test_start = pd.Timestamp('2021-03-10')
assert not data['Date'].isin(purged_dates).any(), 'The five purged dates must remain absent.'
assert test_data['Date'].min() == test_start, 'The fixed test period must begin on 10 March 2021.'
assert set(y_train.unique()) == {0, 1} and set(y_test.unique()) == {0, 1}, \
    'Both classes must exist in the complete training and test periods.'
assert set(existing_test_metrics['model']) == {'DummyClassifier', 'LogisticRegression'}, \
    'The existing clean-test reference must contain the dummy and Logistic Regression models.'

print(f'Predictors: {len(predictors)}')
print(f'Training observations: {len(train_data):,}')
print(f'Test observations: {len(test_data):,}')
print(f'Fixed test start: {test_start.date()}')
print('The target values, chronological assignments and purge are preserved exactly.')

## 2. Pre-specified Random Forest

The Random Forest uses 500 trees, unrestricted tree depth, a minimum of five observations per leaf, square-root feature sampling and balanced subsample class weights. Predictors are passed directly in their original, unscaled values. Every fit call is audited to exclude fixed-test dates and to verify exact equality with the stored predictor values.

In [ ]:
def make_random_forest():
    return RandomForestClassifier(
        n_estimators=500, max_depth=None, min_samples_leaf=5,
        max_features='sqrt', class_weight='balanced_subsample',
        random_state=42, n_jobs=-1,
    )

test_dates = set(test_data['Date'])
fit_audit = []

def audited_fit(model, X_fit, y_fit, dates, label):
    fit_dates = set(pd.DatetimeIndex(dates))
    assert fit_dates.isdisjoint(test_dates), f'Test dates entered model fitting for {label}.'
    pd.testing.assert_frame_equal(
        X_fit, data.loc[X_fit.index, predictors], check_exact=True
    )
    assert not hasattr(model, 'steps'), 'Random Forest must receive unscaled predictors without a pipeline.'
    model.fit(X_fit, y_fit)
    assert len(model.estimators_) == 500, f'{label} did not fit all 500 trees.'
    fit_audit.append({'label': label, 'observations': len(X_fit), 'test_overlap': 0, 'trees': len(model.estimators_)})
    return model

print('The Random Forest specification is fixed in advance; no alternative settings will be searched.')

## 3. Expanding-window temporal-stability diagnostic

Five expanding-window folds with a five-observation gap reproduce the corrected Logistic Regression validation design. Each fold estimates its 75th-percentile target threshold solely from that fold's training `future_rv_5d`, applies it unchanged to training and validation, and fits a fresh 500-tree forest. If validation contains one observed class, accuracy and confusion counts remain available, while balanced accuracy, positive-class precision, recall, F1-score, ROC-AUC and average precision are recorded as missing.

In [ ]:
METRIC_NAMES = [
    'accuracy', 'balanced_accuracy', 'precision', 'recall',
    'f1_score', 'roc_auc', 'average_precision',
]

def calculate_metrics(y_true, predicted_class, probability):
    two_class_sample = set(pd.Series(y_true).unique()) == {0, 1}
    return {
        'accuracy': accuracy_score(y_true, predicted_class),
        'balanced_accuracy': balanced_accuracy_score(y_true, predicted_class) if two_class_sample else np.nan,
        'precision': precision_score(y_true, predicted_class, zero_division=0) if two_class_sample else np.nan,
        'recall': recall_score(y_true, predicted_class, zero_division=0) if two_class_sample else np.nan,
        'f1_score': f1_score(y_true, predicted_class, zero_division=0) if two_class_sample else np.nan,
        'roc_auc': roc_auc_score(y_true, probability) if two_class_sample else np.nan,
        'average_precision': average_precision_score(y_true, probability) if two_class_sample else np.nan,
    }

splitter = TimeSeriesSplit(n_splits=5, gap=5)
cv_rows = []
for fold_number, (fold_train_index, fold_validation_index) in enumerate(splitter.split(X_train), start=1):
    X_fold_train = X_train.iloc[fold_train_index]
    X_fold_validation = X_train.iloc[fold_validation_index]
    fold_training_rv = train_data['future_rv_5d'].iloc[fold_train_index]
    fold_validation_rv = train_data['future_rv_5d'].iloc[fold_validation_index]
    fold_train_dates = train_data['Date'].iloc[fold_train_index]
    fold_validation_dates = train_data['Date'].iloc[fold_validation_index]

    assert fold_train_index.max() + 5 < fold_validation_index.min(), \
        f'Fold {fold_number} must preserve the five-observation gap.'
    assert set(fold_train_dates).isdisjoint(test_dates) and set(fold_validation_dates).isdisjoint(test_dates), \
        f'The fixed test period must not enter fold {fold_number}.'
    fold_threshold = float(fold_training_rv.quantile(0.75))
    threshold_check = float(train_data.loc[fold_training_rv.index, 'future_rv_5d'].quantile(0.75))
    assert np.isclose(fold_threshold, threshold_check, rtol=0, atol=1e-15), \
        f'Fold {fold_number} threshold must use only its training observations.'
    y_fold_train = (fold_training_rv > fold_threshold).astype(int)
    y_fold_validation = (fold_validation_rv > fold_threshold).astype(int)
    assert set(y_fold_train.unique()) == {0, 1}, f'Both training classes must exist in fold {fold_number}.'
    assert y_fold_validation.equals((fold_validation_rv > fold_threshold).astype(int)), \
        f'Fold {fold_number} must apply one fixed training-derived threshold to validation.'

    model = audited_fit(
        make_random_forest(), X_fold_train, y_fold_train, fold_train_dates, f'cv_fold_{fold_number}'
    )
    probability = model.predict_proba(X_fold_validation)[:, 1]
    prediction = (probability >= 0.5).astype(int)
    assert np.logical_and(probability >= 0, probability <= 1).all(), \
        f'Fold {fold_number} probabilities must lie between zero and one.'
    metrics = calculate_metrics(y_fold_validation, prediction, probability)
    tn, fp, fn, tp = confusion_matrix(y_fold_validation, prediction, labels=[0, 1]).ravel()
    cv_rows.append({
        'result_type': 'fold', 'fold': fold_number,
        'training_start': fold_train_dates.min().date().isoformat(),
        'training_end': fold_train_dates.max().date().isoformat(),
        'validation_start': fold_validation_dates.min().date().isoformat(),
        'validation_end': fold_validation_dates.max().date().isoformat(),
        'training_size': len(fold_train_index), 'validation_size': len(fold_validation_index),
        'fold_training_threshold': fold_threshold,
        'training_positive_count': int(y_fold_train.sum()),
        'training_positive_proportion': float(y_fold_train.mean()),
        'validation_positive_count': int(y_fold_validation.sum()),
        'validation_positive_proportion': float(y_fold_validation.mean()),
        'true_negative': int(tn), 'false_positive': int(fp),
        'false_negative': int(fn), 'true_positive': int(tp),
        **metrics,
    })

cv_folds = pd.DataFrame(cv_rows)
logistic_fold_reference = existing_logistic_cv.loc[existing_logistic_cv['result_type'].eq('fold')].copy()
assert np.allclose(
    cv_folds['fold_training_threshold'], logistic_fold_reference['fold_training_threshold'], rtol=0, atol=1e-15
), 'Random Forest and Logistic Regression must use identical fold thresholds.'
assert np.array_equal(
    cv_folds['validation_positive_count'].to_numpy(), logistic_fold_reference['validation_positive_count'].to_numpy()
), 'Validation class counts must match the corrected Logistic Regression design.'

cv_mean = cv_folds[METRIC_NAMES].mean()
cv_std = cv_folds[METRIC_NAMES].std(ddof=1)
cv_valid_count = cv_folds[METRIC_NAMES].notna().sum()
summary_rows = []
for result_type, values in [
    ('mean', cv_mean), ('standard_deviation', cv_std), ('valid_fold_count', cv_valid_count)
]:
    row = {column: np.nan for column in cv_folds.columns}
    row.update({'result_type': result_type, 'fold': np.nan, **values.to_dict()})
    summary_rows.append(row)
rf_cv_table = pd.concat([cv_folds, pd.DataFrame(summary_rows)], ignore_index=True)
available_metrics = cv_folds[METRIC_NAMES].to_numpy(dtype=float)
available_metrics = available_metrics[~np.isnan(available_metrics)]
assert np.isfinite(available_metrics).all(), 'All available CV metrics must be finite.'
assert all(item['test_overlap'] == 0 and item['trees'] == 500 for item in fit_audit), \
    'Every CV fit must exclude test dates and contain 500 trees.'
rf_cv_table.to_csv(RF_CV_PATH, index=False)
print('Random Forest fold thresholds, prevalence and confusion counts:')
print(cv_folds[[
    'fold', 'fold_training_threshold', 'training_positive_count', 'training_positive_proportion',
    'validation_positive_count', 'validation_positive_proportion',
    'true_negative', 'false_positive', 'false_negative', 'true_positive',
]].to_string(index=False, float_format=lambda value: f'{value:.6f}'))
print('Random Forest CV metrics:')
print(cv_folds[['fold', *METRIC_NAMES]].to_string(index=False, float_format=lambda value: f'{value:.6f}'))
print('Random Forest CV summary:')
print(pd.DataFrame({
    'mean': cv_mean, 'standard_deviation': cv_std, 'valid_fold_count': cv_valid_count
}).to_string(float_format=lambda value: f'{value:.6f}'))
print('Fold 1 contains no positive validation observations; only its accuracy and confusion counts are comparable.')

## 4. Unchanged fixed-test evaluation

A fresh Random Forest is fitted on the complete existing training period using its fixed `high_volatility` labels. The unchanged test period is evaluated once at the pre-specified probability threshold of 0.5.

In [ ]:
random_forest = audited_fit(make_random_forest(), X_train, y_train, train_data['Date'], 'final_random_forest')
test_probability = random_forest.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= 0.5).astype(int)
assert np.logical_and(test_probability >= 0, test_probability <= 1).all(), \
    'Fixed-test probabilities must lie between zero and one.'
assert len(random_forest.estimators_) == 500, 'The final Random Forest must contain all 500 trees.'
metrics = calculate_metrics(y_test, test_prediction, test_probability)
tn, fp, fn, tp = confusion_matrix(y_test, test_prediction, labels=[0, 1]).ravel()
rf_test_table = pd.DataFrame([{
    'model': 'Random Forest', 'classification_threshold': 0.5, **metrics,
    'true_negative': int(tn), 'false_positive': int(fp),
    'false_negative': int(fn), 'true_positive': int(tp),
}])
assert np.isfinite(rf_test_table.drop(columns='model').to_numpy(dtype=float)).all(), \
    'All fixed-test metrics must be finite.'
assert all(item['test_overlap'] == 0 and item['trees'] == 500 for item in fit_audit), \
    'No fixed-test date may enter fitting, and every fit must contain 500 trees.'
pd.testing.assert_frame_equal(data[['Date', target, 'sample_period']], original_assignments)
rf_test_table.to_csv(RF_TEST_PATH, index=False)
print('Random Forest fixed-test metrics and confusion counts:')
print(rf_test_table.to_string(index=False, float_format=lambda value: f'{value:.6f}'))

## 5. Clean-test model comparison

The dummy and Logistic Regression rows are reproduced directly from the existing clean-test table rather than refitted. Comparison emphasises balanced accuracy, F1-score, ROC-AUC and average precision; ordinary accuracy alone is insufficient under class imbalance.

In [ ]:
comparison = existing_test_metrics.copy(deep=True)
comparison['model'] = comparison['model'].replace({
    'DummyClassifier': 'Dummy Classifier', 'LogisticRegression': 'Logistic Regression'
})
comparison = pd.concat([comparison, rf_test_table], ignore_index=True)
expected_existing = existing_test_metrics.copy(deep=True)
reproduced_existing = comparison.iloc[:2].copy()
reproduced_existing['model'] = reproduced_existing['model'].replace({
    'Dummy Classifier': 'DummyClassifier', 'Logistic Regression': 'LogisticRegression'
})
pd.testing.assert_frame_equal(
    reproduced_existing.reset_index(drop=True), expected_existing.reset_index(drop=True), check_exact=True
)
comparison.to_csv(COMPARISON_PATH, index=False)
print('Clean fixed-test comparison:')
print(comparison[[
    'model', 'accuracy', 'balanced_accuracy', 'f1_score', 'roc_auc', 'average_precision'
]].to_string(index=False, float_format=lambda value: f'{value:.6f}'))
print('Model interpretation should not rely on ordinary accuracy alone.')

## 6. Impurity-based feature importance

Random Forest impurity importance is descriptive rather than causal. Correlated predictors can divide or distort importance, and these values must not be treated as definitive feature selection.

In [ ]:
importance_table = pd.DataFrame({
    'feature_name': predictors, 'importance': random_forest.feature_importances_,
}).merge(
    feature_dictionary[['feature_name', 'feature_group']], on='feature_name', how='left', validate='one_to_one'
)
importance_table = importance_table[[
    'feature_name', 'feature_group', 'importance'
]].sort_values('importance', ascending=False, ignore_index=True)
assert np.isclose(importance_table['importance'].sum(), 1.0), 'Feature importances must sum to one.'
assert np.isfinite(importance_table['importance']).all(), 'Feature importances must be finite.'
importance_table.to_csv(IMPORTANCE_PATH, index=False)
print('Impurity-based Random Forest feature importance:')
print(importance_table.to_string(index=False, float_format=lambda value: f'{value:.6f}'))
print('Warning: correlation can divide or distort impurity importance; it is descriptive, not causal.')

## 7. Diagnostic figures

The figures report the fixed-test Random Forest confusion matrix, a multi-metric clean-test comparison and descriptive impurity importance.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(
    confusion_matrix=np.array([[tn, fp], [fn, tp]]),
    display_labels=['Normal volatility (0)', 'High volatility (1)'],
).plot(ax=ax, cmap='Greens', colorbar=False, values_format='d')
ax.set_title('XLK Random Forest Test Confusion Matrix')
ax.set_xlabel('Predicted class')
ax.set_ylabel('Actual class')
fig.tight_layout()
fig.savefig(CONFUSION_PATH, dpi=300, bbox_inches='tight')
plt.show()

comparison_metrics = ['accuracy', 'balanced_accuracy', 'f1_score', 'roc_auc', 'average_precision']
comparison_plot = comparison.set_index('model')[comparison_metrics]
fig, ax = plt.subplots(figsize=(12, 7))
comparison_plot.plot(kind='bar', ax=ax, width=0.75)
ax.set_title('XLK Clean-Test Model Comparison')
ax.set_xlabel('Model')
ax.set_ylabel('Metric value')
ax.set_ylim(0, 1)
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Metric', loc='lower right')
fig.tight_layout()
fig.savefig(COMPARISON_FIGURE_PATH, dpi=300, bbox_inches='tight')
plt.show()

plot_importance = importance_table.sort_values('importance', ascending=True)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(plot_importance['feature_name'], plot_importance['importance'], color='#4c956c')
ax.set_title('XLK Random Forest Impurity-Based Feature Importance')
ax.set_xlabel('Impurity-based importance')
ax.set_ylabel('Predictor')
ax.set_xlim(left=0)
fig.tight_layout()
fig.savefig(IMPORTANCE_FIGURE_PATH, dpi=300, bbox_inches='tight')
plt.show()

print('Saved all Random Forest baseline tables and figures.')
print('All quality checks passed; no tuning, scaling, noise or test-set fitting was performed.')

## Interpretation and limitations

This fixed specification provides a nonlinear clean-data comparison rather than an optimised model. Results should be interpreted using balanced accuracy, F1-score, ROC-AUC and average precision alongside ordinary accuracy. Temporal variation in fold prevalence, overlapping adjacent targets, class weighting and correlated predictors remain important limitations. Impurity importance is descriptive and does not establish causality or justify automatic feature removal.